# Film Db Rag

Generated from spec: film_db_rag.yaml

In [ ]:
import os, json, requests
from pathlib import Path

# Optionally load .env written by initContainer into /home/jovyan/work/.rag/.env
envfile = Path('/home/jovyan/work/.rag/.env')
if envfile.exists():
    for line in envfile.read_text().splitlines():
        if '=' in line and not line.strip().startswith('#'):
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

PROXY_HOST = os.environ.get('PROXY_HOST', '')
PROXY_BASE = os.environ.get('PROXY_BASE') or (f"https://{PROXY_HOST}" if PROXY_HOST else '')
SCHEMA_NAME = os.environ.get('SCHEMA_NAME', 'public')

USER_EMAIL = os.environ.get('USER_EMAIL', '')
STORAGE_ACCESS_KEY_ID = os.environ.get('STORAGE_ACCESS_KEY_ID', '')
STORAGE_SECRET_ACCESS_KEY = os.environ.get('STORAGE_SECRET_ACCESS_KEY', '')

LLMAPI_BASE = os.environ.get('LLMAPI_BASE', 'https://llmapi6.llmosaic.ai')
LLMAPI_MODEL_NAMES = os.environ.get('LLMAPI_MODEL_NAMES', '')
LLMAPI_COMPLETION_MODEL = os.environ.get('LLMAPI_COMPLETION_MODEL', (LLMAPI_MODEL_NAMES.split(',')[0] if LLMAPI_MODEL_NAMES else 'gpt-oss-120b'))
LLMAPI_EMBED_MODEL = os.environ.get('LLMAPI_EMBED_MODEL', 'titan-embed-text-v2')
LLMAPI_VECTOR_DIMENSION = int(os.environ.get('LLMAPI_VECTOR_DIMENSION', '1024'))
LLM_NAME = LLMAPI_COMPLETION_MODEL
EMBED_NAME = LLMAPI_EMBED_MODEL

assert PROXY_BASE, 'Set PROXY_BASE or PROXY_HOST in env/.env'

AUTH_BEARER = f"{STORAGE_ACCESS_KEY_ID}:{STORAGE_SECRET_ACCESS_KEY}:storage" if STORAGE_ACCESS_KEY_ID and STORAGE_SECRET_ACCESS_KEY else ''
PROXY_HEADERS = {'Authorization': f'Bearer {AUTH_BEARER}'} if AUTH_BEARER else {}
LLM_HEADERS = {
    'Authorization': f"Bearer {os.environ.get('LLMAPI_API_KEY', '')}",
    'Content-Type': 'application/json'
}
_embed_key = (os.environ.get('LLMAPI_EMBED_KEY') or os.environ.get('LLMAPI_API_KEY') or '')
EMBED_HEADERS = {
    'Authorization': f"Bearer {_embed_key}",
    'Content-Type': 'application/json'
}
print('Embed token present:', bool(_embed_key))

print('Using PROXY_BASE=', PROXY_BASE)
print('Using SCHEMA_NAME=', SCHEMA_NAME)

In [ ]:
# Scoped variable from spec: prefer example's default_schema over global .env
SCHEMA_NAME = 'filmdata1'
print('Overriding SCHEMA_NAME from spec.default_schema ->', SCHEMA_NAME)

## Health Check

Verify the tenant PostgREST-Proxy is reachable and ready.

In [ ]:
url = PROXY_BASE + "/healthz"
r = requests.get(url, headers=PROXY_HEADERS)
print(r.status_code)
print(r.text[:2000])

## Drop Schema (if exists)

Ensure a clean schema before restore.

In [ ]:
url = PROXY_BASE + "/drop-schema"
body = {"schema_name": SCHEMA_NAME, "if_exists": True}
r = requests.post(url, headers=dict(**PROXY_HEADERS, **{'Content-Type':'application/json'}), json=body)
print(r.status_code, r.text)

## Restore Film DB

Restore the film database from the preloaded SQL fixture copied into Jupyter examples.

In [ ]:
path = "/home/jovyan/work/examples/film_db_backup.sql"
url = PROXY_BASE + "/restore-backup?schemaName=" + SCHEMA_NAME
with open(path, 'rb') as fh:
    files = {'backup_file': ('film_db_backup.sql', fh, 'application/sql')}
    r = requests.post(url, headers=PROXY_HEADERS, files=files)
print(r.status_code)
print(r.text[:2000])
try:
    jr = r.json()
except Exception:
    jr = {}
# If job was accepted (async), poll for artifact log readiness
if isinstance(jr, dict) and jr.get('status') == 'accepted' and jr.get('artifact_url'):
    import time
    artifact_url = PROXY_BASE + str(jr['artifact_url'])
    print('Polling for restore completion:', artifact_url)
    t0=time.time(); timeout=180; interval=3
    while True:
        rr = requests.get(artifact_url, headers=PROXY_HEADERS)
        if rr.status_code == 200:
            print('Restore log available (HTTP 200).')
            print(rr.text[-1000:])
            break
        if time.time()-t0 > timeout:
            print('Timed out waiting for restore completion log.')
            break
        time.sleep(interval)

## Prepare Film Texts

Extract a small set of film descriptions for embedding.

In [ ]:
url = PROXY_BASE + "/film_list?limit=10"
r = requests.get(url, headers=dict(**PROXY_HEADERS, **{'Accept-Profile': SCHEMA_NAME}))
print(r.status_code)
try:
    j = r.json()
    FILM_DOCS = [row.get('description', '') for row in (j or [])]
    FILM_IDS = [row.get('id') for row in (j or [])]
    print('prepared', len(FILM_DOCS), 'film docs')
except Exception as e:
    print('failed to parse film list:', e)

## Drop Embeddings Table (if exists)

Ensure a clean state for the embeddings table.

In [ ]:
url = PROXY_BASE + "/drop-table?schemaName=" + SCHEMA_NAME
body = {"table_name": "film_embeddings", "if_exists": True}
r = requests.post(url, headers=dict(**PROXY_HEADERS, **{'Content-Type':'application/json'}), json=body)
print(r.status_code, r.text)

## Create Embeddings Table

Create a table to store film texts and embeddings.

In [ ]:
url = PROXY_BASE + "/create-table?schemaName=" + SCHEMA_NAME
body = {"table_name": "film_embeddings", "not_exists": True, "columns": [{"name": "id", "type": "bigserial", "constraints": "PRIMARY KEY"}, {"name": "film_id", "type": "integer"}, {"name": "document_text", "type": "text"}, {"name": "embedding", "type": "vector(1024)"}]}
r = requests.post(url, headers=dict(**PROXY_HEADERS, **{'Content-Type':'application/json'}), json=body)
print(r.status_code, r.text)

## Create Vector Index (HNSW)

Build a HNSW cosine index on the embedding column for fast similarity search.

In [ ]:
url = PROXY_BASE + "/create-vector-index?schemaName=" + SCHEMA_NAME
body = {"table_name": "film_embeddings", "vector_column": "embedding", "index_type": "hnsw", "distance_operator": "vector_cosine_ops"}
r = requests.post(url, headers=dict(**PROXY_HEADERS, **{'Content-Type':'application/json'}), json=body)
print(r.status_code, r.text)

## Embed + Insert Films

Generate embeddings for selected film descriptions and insert rows.

In [ ]:
if 'FILM_DOCS' not in globals() or not FILM_DOCS:
    # Fallback: fetch film_list and prepare docs/ids
    import json as _json
    _fr = requests.get(PROXY_BASE + "/film_list?limit=" + str(10), headers=dict(**PROXY_HEADERS, **{'Accept-Profile': SCHEMA_NAME}))
    try:
        _rows = _fr.json() if _fr.ok else []
    except Exception:
        _rows = []
    FILM_DOCS = []
    FILM_IDS = []
    for _r in (_rows or []):
        _id = int(_r.get('film_id') or _r.get('id') or _r.get('fid') or 0)
        _title = str(_r.get('title') or 'Untitled')
        _year = (" (" + str(_r.get('release_year')) + ")" if _r.get('release_year') else '')
        _desc = str(_r.get('description') or _r.get('plot') or _r.get('overview') or '')
        _doc = (_title + _year + " — " + _desc).strip()
        FILM_DOCS.append(_doc)
        FILM_IDS.append(_id)
assert 'FILM_DOCS' in globals() and FILM_DOCS, "Run 'Prepare Film Texts' step first"
for i, t in enumerate(FILM_DOCS):
    fid = (FILM_IDS[i] if 'FILM_IDS' in globals() and i < len(FILM_IDS) else i+1)
    er = requests.post(LLMAPI_BASE + "/" + EMBED_NAME + "/v1/embeddings", headers=EMBED_HEADERS, json={"model": EMBED_NAME, "input": [t]})
    if er.status_code != 200:
        print('embed error', er.status_code, er.text[:500])
        continue
    try:
        ej = er.json()
    except Exception as e:
        print('embed parse error', str(e), er.text[:500])
        continue
    vec = (ej.get('data') or [{}])[0].get('embedding')
    ir = requests.post(PROXY_BASE + "/film_embeddings", headers=dict(**PROXY_HEADERS, **{'Content-Type':'application/json', 'Content-Profile': SCHEMA_NAME}), json={"film_id": fid, "document_text": t, "embedding": vec})
    print('insert', fid, ir.status_code)

## Vector Query

Embed the query text and find the closest film by cosine distance.

In [ ]:
er = requests.post(LLMAPI_BASE + "/" + EMBED_NAME + "/v1/embeddings", headers=EMBED_HEADERS, json={"model": EMBED_NAME, "input": ["Adventure on the open sea with pirates"]})
if er.status_code != 200:
    print('embed error', er.status_code, er.text[:500])
try:
    ej = er.json(); vec = (ej.get('data') or [{}])[0].get('embedding')
except Exception as e:
    print('embed parse error', str(e), er.text[:500]); vec = None
if not vec:
    print('no embedding vector produced; check tokens and model name'); vec = []
encoded = requests.utils.quote(json.dumps(vec))
qr = requests.get(PROXY_BASE + "/film_embeddings?query_vector=" + encoded + "&vector_column=embedding&distance_operator=<=>&limit=3", headers=dict(**PROXY_HEADERS, **{'Accept-Profile': SCHEMA_NAME}))
print(qr.status_code)
print(qr.text)
try:
    LAST_RESULTS = qr.json()
except Exception:
    LAST_RESULTS = []

## Chat with Retrieved Context

Call the chat model using the retrieved synopsis as grounding context.

In [ ]:
ctx = ''
try:
    if isinstance(LAST_RESULTS, list) and LAST_RESULTS:
        first = LAST_RESULTS[0]
        ctx = first.get('data', {}).get('text') or first.get('document_text') or ''
except Exception:
    pass
template = "Using the following film synopsis as context: \"{context}\", answer the question: \"{question}\"\n"
prompt = template.format(context=ctx, question="Which characters are central to the story?")
cr = requests.post(LLMAPI_BASE + "/" + LLM_NAME + "/v1/chat/completions", headers=LLM_HEADERS, json={"model": LLM_NAME, "messages": [{"role": "user", "content": prompt}], "max_tokens": 256, "temperature": 0.7})
print(cr.status_code)
print(cr.text)